## Fase #2: Comprensión de los Datos / Data Understanding
La fase de comprensión de datos comienza con una recopilación inicial de datos y continúa con actividades para familiarizarse con los datos, identificar problemas de calidad de los datos, descubrir los primeros conocimientos sobre los datos o detectar subconjuntos interesantes para formular hipótesis sobre información oculta.
* Establecé una metodología de adquisición de los datos para el negocio.
* Contemplá en la adquisición de datos una alternativa donde debas construir el setde datos.
* Identificá los distintos problemas o desafíos con los datos (datos faltantes, ruidos,etc) que puedan afectar a su calidad.
* Enumerá las distintas fases, tareas generales, tareas específicas e instancias del proceso que incluirás como estrategia para aplicar la metodología

### Adquisición, construcción y calidad

**Fuente principal.** Se descarga manualmente el CSV de actualización diaria del Adjunto IV y se conserva sin editar. Cada ejecución registra nombre, tamaño, SHA-256 y fecha máxima de `fecha_data`; así se puede identificar exactamente la versión utilizada.

**Alternativa construida.** Como extensión, se puede unir el Adjunto IV con la producción no convencional mediante `idpozo` y agregar la producción de los primeros seis meses posteriores a `fecha_fin_fractura`. Ese conjunto permitiría estudiar producción inicial según características de completación. Requiere resolver pozos con múltiples intervenciones, meses faltantes y ventanas incompletas, por lo que no forma parte del modelo principal.

**Desafíos de calidad.** Se controlarán duplicados de `id_base_fractura_adjiv`, valores no numéricos, ceros, fechas inválidas o invertidas, categorías poco frecuentes, pozos repetidos y valores extremos. Los filtros se aplican de manera secuencial para que el resumen de exclusiones sea reproducible. No se imputará la variable objetivo.

In [ ]:
import hashlib
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display  # ty: ignore[unresolved-import]


In [ ]:
# @title Obtener el dataset en Colab/repositorio

dataset_url = "http://datos.energia.gob.ar/dataset/71fa2e84-0316-4a1b-af68-7f35e41f58d7/resource/2280ad92-6ed3-403e-a095-50139863ab0d/download/datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv"

NOMBRE_ARCHIVO = "datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv"

dataset_repo_path = Path("data") / NOMBRE_ARCHIVO
dataset_colab_path = Path("/content/data") / NOMBRE_ARCHIVO

if dataset_repo_path.exists():
	DATA_PATH = dataset_repo_path
elif dataset_colab_path.exists():
	DATA_PATH = dataset_colab_path
else:
	dataset_repo_path.parent.mkdir(parents=True, exist_ok=True)
	urlretrieve(dataset_url, dataset_repo_path)
	DATA_PATH = dataset_repo_path

print(f"Dataset disponible en: {DATA_PATH}")

Dataset disponible en: data\datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv


In [ ]:
# @title Carga de dataframe

df_raw = pd.read_csv(DATA_PATH)

In [ ]:
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)


sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

columnas_numericas = [
	"longitud_rama_horizontal_m",
	"cantidad_fracturas",
	"agua_inyectada_m3",
	"arena_bombeada_nacional_tn",
	"arena_bombeada_importada_tn",
	"co2_inyectado_m3",
	"presion_maxima_psi",
	"potencia_equipos_fractura_hp",
]
columnas_fecha = ["fecha_inicio_fractura", "fecha_fin_fractura", "fecha_data"]

df = df_raw.copy()
for col in columnas_numericas:
	df[col] = pd.to_numeric(df[col], errors="coerce")
for col in columnas_fecha:
	df[col] = pd.to_datetime(df[col], errors="coerce")

pasos = []


def aplicar_filtro(nombre, condicion):
	global df
	antes = len(df)
	df = df.loc[condicion].copy()
	pasos.append({"paso": nombre, "conservados": len(df), "excluidos_en_paso": antes - len(df)})


pasos.append({"paso": "Archivo original", "conservados": len(df), "excluidos_en_paso": 0})
aplicar_filtro("Cuenca Neuquina", df["cuenca"].eq("NEUQUINA"))
aplicar_filtro("Reservorio no convencional", df["tipo_reservorio"].eq("NO CONVENCIONAL"))
aplicar_filtro("Rama horizontal > 0", df["longitud_rama_horizontal_m"].gt(0))
aplicar_filtro("Cantidad de fracturas > 0", df["cantidad_fracturas"].gt(0))
aplicar_filtro("Agua inyectada > 0", df["agua_inyectada_m3"].gt(0))
aplicar_filtro("Fechas de operación válidas", df[["fecha_inicio_fractura", "fecha_fin_fractura"]].notna().all(axis=1))
aplicar_filtro("Fin no anterior al inicio", df["fecha_fin_fractura"].ge(df["fecha_inicio_fractura"]))

duplicados_id = int(df["id_base_fractura_adjiv"].duplicated().sum())
if duplicados_id:
	raise ValueError(f"Hay {duplicados_id} identificadores de fractura duplicados; deben revisarse.")

perfil_fuente = pd.DataFrame(
	{
		"indicador": ["archivo", "tamaño_bytes", "sha256", "filas_originales", "columnas", "fecha_data_máxima", "pozos_en_muestra"],
		"valor": [
			DATA_PATH.name,
			DATA_PATH.stat().st_size,
			sha256,
			len(df_raw),
			df_raw.shape[1],
			df_raw["fecha_data"].pipe(pd.to_datetime, errors="coerce").max(),
			df["idpozo"].nunique(),
		],
	}
)
display(perfil_fuente)
display(pd.DataFrame(pasos))
print(f"Conjunto analítico: {len(df):,} registros de {df['idpozo'].nunique():,} pozos.")

,indicador,valor
0,archivo,datos-de-fractura-de-pozos-de-hidrocarburos-ad...
1,tamaño_bytes,1255065
2,sha256,72b5204463dff0f8a4f951f893344abb8992a540caa8fc...
3,filas_originales,4983
4,columnas,30
5,fecha_data_máxima,2026-10-07 08:43:27.073465
6,pozos_en_muestra,2666


,paso,conservados,excluidos_en_paso
0,Archivo original,4983,0
1,Cuenca Neuquina,4180,803
2,Reservorio no convencional,3805,375
3,Rama horizontal > 0,2734,1071
4,Cantidad de fracturas > 0,2734,0
5,Agua inyectada > 0,2732,2
6,Fechas de operación válidas,2732,0
7,Fin no anterior al inicio,2700,32


Conjunto analítico: 2,700 registros de 2,666 pozos.


### Interpretación de la adquisición y la calidad

El archivo analizado contiene **4.910 registros y 30 variables**. La huella SHA-256 permite reconocer esta versión exacta del CSV, cuya fecha máxima de carga es el **29/09/2026**. Después de aplicar todos los criterios quedan **2.626 registros de 2.592 pozos**. La diferencia entre registros y pozos confirma que algunos pozos tienen más de una intervención; por eso `idpozo` se conserva como control y no como variable predictora.

Los filtros son secuenciales: `excluidos_en_paso` se interpreta respecto del paso anterior, no del archivo completo. Primero se retiraron 803 registros de otras cuencas y luego 376 correspondientes a reservorios convencionales dentro de la cuenca Neuquina. La mayor reducción posterior proviene de **1.071 registros sin una rama horizontal positiva**, coherente con el alcance limitado a pozos horizontales. También se excluyeron dos registros sin agua positiva y 32 con fecha de finalización anterior a la fecha de inicio.

El conjunto final representa aproximadamente el **53,5 %** del archivo original. Esta reducción no implica por sí sola mala calidad: una parte importante queda afuera por definición del alcance. En cambio, las fechas invertidas sí constituyen un problema de consistencia. Los resultados sólo describen el subconjunto retenido y no deben generalizarse a pozos convencionales, verticales u otras cuencas.